In [5]:
tabla='ctcam10'
col_essi='fec_cit'
col_tabla='citambproconfec'
essi='essi_dat_cex001_etl'

In [6]:
from decouple import config
import decouple
from sqlalchemy import create_engine
import pandas as pd
import oracledb
import sys
from sqlalchemy import text

In [7]:
#CONEXIONES
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [8]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=2", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=2"

c1= text(query)
connection2.execute(c1)

2023-05-01


In [9]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
cas_red.to_sql(name='tmp_cas_red', con=engine1, if_exists='replace', index=False)

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios.to_sql(name='tmp_servicios', con=engine1, if_exists='replace', index=False)

areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
areas.to_sql(name='tmp_areas', con=engine1, if_exists='replace', index=False)

subacti = pd.read_sql_query(f"SELECT cod_sub,des_sub FROM dim_subacti", con=connection2)
subacti.to_sql(name='tmp_subacti', con=engine1, if_exists='replace', index=False)


activi = pd.read_sql_query(f"SELECT cod_act,des_act FROM dim_activi", con=connection2)
activi.to_sql(name='tmp_activi', con=engine1, if_exists='replace', index=False)


cexestsolcit = pd.read_sql_query(f"SELECT cod_esc,des_esc FROM dim_cexestsolcit", con=connection2)
cexestsolcit.to_sql(name='tmp_cexestsolcit', con=engine1, if_exists='replace', index=False)


cexcitrep = pd.read_sql_query(f"SELECT cod_cre,des_cre FROM dim_cexcitrep", con=connection2)
cexcitrep.to_sql(name='tmp_cexcitrep', con=engine1, if_exists='replace', index=False)


tipcita = pd.read_sql_query(f"SELECT cod_tci,des_tci FROM dim_tipcita", con=connection2)
tipcita.to_sql(name='tmp_tipcita', con=engine1, if_exists='replace', index=False)









tipemi = pd.read_sql_query(f"SELECT cod_emi,des_emi FROM dim_tipemi", con=connection2)
tipemi.to_sql(name='tmp_tipemi', con=engine1, if_exists='replace', index=False)

cexmotelicit = pd.read_sql_query(f"SELECT cod_eli,des_eli FROM dim_cexmotelicit", con=connection2)
cexmotelicit.to_sql(name='tmp_cexmotelicit', con=engine1, if_exists='replace', index=False)

estcit = pd.read_sql_query(f"SELECT cod_eci,des_eci FROM dim_estcit", con=connection2)
estcit.to_sql(name='tmp_estcit', con=engine1, if_exists='replace', index=False)

cexcitoto = pd.read_sql_query(f"SELECT cod_eco,des_eco FROM dim_cexcitoto", con=connection2)
cexcitoto.to_sql(name='tmp_cexcitoto', con=engine1, if_exists='replace', index=False)


12

query_count_before = f"SELECT COUNT(*) FROM {essi}"
cant_antes = connection1.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} antes de la inserción: {cant_antes}")

In [ ]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [ ]:
query1=f"""

INSERT INTO {essi} 
(cod_act,cod_sub,fec_anu,cod_are,cod_cas,cnv_esp,fec_cre,ctr_seg,hor_cit,ip_anu,ip_cre,ip_mod,mod_anu,fec_mod,cit_num,ord_cit,ori_cas,pac_sec,num_doc,cas_pro,fec_cit,ori_pro,hor_ter,hor_ini,cit_rep,cod_ser,fec_sol,cod_tdi,usu_anu,usu_cre,usu_mod,cod_cci,cod_eci,cod_enco,cod_oto,cod_mec,mot_cit,cod_tci) 

SELECT 
citambactcod,citambactespcod,citambanufec,citambarehoscod,citambcenasicod,citambcnvespcod,citambcrefec,citambestctrlseg,citambhorcit,citambipanucod,citambipcrecod,citambipmodcod,citambmodanucod,citambmodfec,citambnum,citambnumord,citamboricenasicod,citambpacsecnum,citambperasisdocidennum,citambproconcenasicod,citambproconfec,citambproconoricenasicod,citambproconturhorfin,citambproconturhorini,citambrep,citambservhoscod,citambsolfec,citambtipdocidenpercod,citambusuanucod,citambusucrecod,citambusumodcod,condcitacod,estcitcod,estcitotocod,modotorcitacod,motelicitcod,motinacitdes,tipocitacod





FROM dblink('dbname=dl_essi user=ugaddba001ir password=U64dING23', 'SELECT * FROM {tabla} WHERE {tabla}.{col_tabla} >=''{fecha}''')

AS tmp_tbl(
	citamboricenasicod  character(1),
	citambcenasicod  character(3),
	citambnum  numeric(10,0),
	citambproconoricenasicod  character(1),
	citambproconcenasicod  character(3),
	citambarehoscod  character(2),
	citambservhoscod  character(3),
	citambactcod  character(2),
	citambactespcod  character(3),
	citambtipdocidenpercod  character(1),
	citambperasisdocidennum  character(10),
	citambproconfec  timestamp,
	citambproconturhorini  timestamp,
	citambproconturhorfin  timestamp,
	tipocitacod  character(1),
	condcitacod  character(1),
	modotorcitacod  character(1),
	citambnumord  numeric(3,0),
	citambhorcit  timestamp,
	citambrep  character(1),
	motelicitcod  character(2),
	estcitcod  character(1),
	estcitotocod  character(1),
	citambusucrecod  character(10),
	citambcrefec  timestamp,
	citambusumodcod  character(10),
	citambmodfec  timestamp,
	citambsolfec  timestamp,
	citambipcrecod  character(15),
	citambipmodcod  character(15),
	motinacitdes  character(110),
	citambpacsecnum  numeric(10,0),
	citambusuanucod  character(10),
	citambanufec  date,
	citambipanucod  character(15),
	citambmodanucod  character(1),
	citambestctrlseg  character(1),
	citambcnvespcod  numeric(3,0)
);
"""
c2= text(query1)
cursor=connection1.execute(c2)




query_count_after = f"SELECT COUNT(*) FROM {essi}"
cant_despues = connection1.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

### Trayendo maestros

In [ ]:
query1=f"""
ALTER TABLE tmp_cas_red 
ALTER COLUMN id_red TYPE CHARACTER (2),
ALTER COLUMN cod_cas TYPE CHARACTER (3),
ALTER COLUMN cod_red TYPE CHARACTER (2),
ALTER COLUMN des_red TYPE CHARACTER (60),
ALTER COLUMN des_cas TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_cas= t.des_cas,
des_red= t.des_red,
cod_red= t.cod_red
FROM tmp_cas_red t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_cas = t.cod_cas and {essi}.cod_cas IS NOT NULL and t.cod_cas IS NOT NULL ;



ALTER TABLE tmp_servicios
ALTER COLUMN cod_ser TYPE CHARACTER (3),
ALTER COLUMN des_ser TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_ser= t.des_ser

FROM tmp_servicios t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_ser = t.cod_ser and {essi}.cod_ser IS NOT NULL and t.cod_ser IS NOT NULL ;



ALTER TABLE tmp_areas
ALTER COLUMN cod_are TYPE CHARACTER (2),
ALTER COLUMN des_are TYPE CHARACTER (30);

UPDATE {essi} 
SET 
des_are= t.des_are

FROM tmp_areas t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_are = t.cod_are and {essi}.cod_are IS NOT NULL and t.cod_are IS NOT NULL ;




ALTER TABLE tmp_activi 
ALTER COLUMN cod_act TYPE CHARACTER (2),
ALTER COLUMN des_act TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_act= t.des_act

FROM tmp_activi t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_act = t.cod_act and {essi}.cod_act IS NOT NULL and t.cod_act IS NOT NULL ;


ALTER TABLE tmp_subacti
ALTER COLUMN cod_sub TYPE CHARACTER (3),
ALTER COLUMN des_sub TYPE CHARACTER (60);

UPDATE {essi} 
SET 
des_sub= t.des_sub

FROM tmp_subacti t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_sub = t.cod_sub;



"""
c2= text(query1)
cursor=connection1.execute(c2)

In [6]:
query1=f"""


ALTER TABLE tmp_cexestsolcit
ALTER COLUMN cod_esc TYPE CHARACTER (1),
ALTER COLUMN des_esc TYPE CHARACTER (25);

UPDATE {essi} 
SET 
des_tci= t.des_esc

FROM tmp_cexestsolcit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_tci = t.cod_esc;



ALTER TABLE tmp_cexcitrep
ALTER COLUMN cod_cre TYPE CHARACTER (1),
ALTER COLUMN des_cre TYPE CHARACTER (2);

UPDATE {essi} 
SET 
des_rep= t.des_cre

FROM tmp_cexcitrep t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cit_rep = t.cod_cre;



ALTER TABLE tmp_tipcita
ALTER COLUMN cod_tci TYPE CHARACTER (1),
ALTER COLUMN des_tci TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_cci= t.des_tci

FROM tmp_tipcita t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_cci = t.cod_tci;




"""
c2= text(query1)
cursor=connection1.execute(c2)



In [7]:
query1=f"""


ALTER TABLE tmp_tipemi
ALTER COLUMN cod_emi TYPE CHARACTER (1),
ALTER COLUMN des_emi TYPE CHARACTER (20);

UPDATE {essi} 
SET 
des_oto= t.des_emi

FROM tmp_tipemi t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_oto = t.cod_emi;



ALTER TABLE tmp_cexmotelicit
ALTER COLUMN cod_eli TYPE CHARACTER (2),
ALTER COLUMN des_eli TYPE CHARACTER (30);

UPDATE {essi} 
SET 
des_mec= t.des_eli

FROM tmp_cexmotelicit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_mec = t.cod_eli;



ALTER TABLE tmp_estcit
ALTER COLUMN cod_eci TYPE CHARACTER (1),
ALTER COLUMN des_eci TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_eci= t.des_eci

FROM tmp_estcit t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_eci = t.cod_eci;




ALTER TABLE tmp_cexcitoto
ALTER COLUMN cod_eco TYPE CHARACTER (1),
ALTER COLUMN des_eco TYPE CHARACTER (15);

UPDATE {essi} 
SET 
des_enco= t.des_eco

FROM tmp_cexcitoto t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_enco = t.cod_eco;






DROP TABLE tmp_areas;
DROP TABLE tmp_servicios;
DROP TABLE tmp_cas_red;
DROP TABLE tmp_activi;
DROP TABLE tmp_subacti;
DROP TABLE tmp_cexestsolcit;
DROP TABLE tmp_cexcitrep;
DROP TABLE tmp_tipcita;
DROP TABLE tmp_tipemi;
DROP TABLE tmp_cexmotelicit;
DROP TABLE tmp_estcit;
DROP TABLE tmp_cexcitoto;


"""
c2= text(query1)
cursor=connection1.execute(c2)



In [ ]:
connection1.close()
connection2.close()
connection3.close()